# Probabilistic glucose trajectories with Flow Matching Neural Processes

**Tutorial** · Álvaro Inclán Balán, Enrique Bravo Díaz, Jaime Morata Bermúdez (UAM, Máster en IA Generativa)

Continuous glucose monitors (CGM) record glucose every 5 minutes, but the series have gaps. In this tutorial we show, step by step, how we model these series as **random functions** with neural processes and how FlowNP compares with a classical neural process.

We will:

1. Download the GlucoBench data and build 6-hour windows and fixed evaluation tasks.
2. Look at what a *task* is: context points, target points and a gap.
3. Train a small **FlowNP** and a **CNP** for a few minutes.
4. Draw sample trajectories from both models and compare them.
5. Score them with the shared metrics (LL, RMSE, CRPS, coverage).
6. Load the results of the full experiments (100k steps, 3 seeds) and reproduce the main tables and figures.

It runs on Google Colab (*Runtime → Change runtime type → T4 GPU*, about 10 minutes) or locally on a CPU, no GPU needed (about 12 minutes on a 4-core laptop CPU, most of it in section 6).

## 1. Setup

The notebook runs on Google Colab or locally (Jupyter, Python ≥ 3.9 with PyTorch installed). If it is opened from a clone of the repository it uses that clone; otherwise (e.g. on Colab) it clones it. It then installs the two packages that are usually missing: `flow-matching` (Meta's flow matching library, used by FlowNP) and `attrdictionary` (used by the neural process code).

In [ ]:
REPO_URL = 'https://github.com/enriquebravo22/FlowNP-glucose.git'

import os, sys, subprocess

def run(cmd, cwd=None):
    ''' Run a command and show its output in the notebook (stops on error). '''
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-3000:])
    r.check_returncode()

if os.path.isdir('glucose_np') and os.path.isdir('flowNP'):
    REPO = os.getcwd()                                   # opened from a local clone
else:
    if not os.path.isdir('FlowNP-glucose'):
        run(['git', 'clone', '-q', REPO_URL, 'FlowNP-glucose'])
    REPO = os.path.abspath('FlowNP-glucose')
run([sys.executable, '-m', 'pip', 'install', '-q', 'flow-matching==1.0.10', 'attrdictionary'])
os.chdir(REPO)
print('repository:', REPO)

In [ ]:
import time, json, glob, shutil, zipfile, tempfile
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(REPO, 'glucose_np'))   # data, tasks and metrics (shared by all models)
sys.path.insert(0, os.path.join(REPO, 'flowNP'))       # models and training scripts

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name() if device == 'cuda' else '')

## 2. Data: from raw CGM readings to tasks

[GlucoBench](https://github.com/IrinaStatsLab/GlucoBench) collects five public CGM datasets (Broll, Colas, Dubosson, Hall and Weinstock; 477 patients). The raw CSVs are not redistributed in our repository, so we download them from the GlucoBench repository.

The pipeline in `glucose_np/` then:

* `01_preprocess.py` clips readings to the sensor range [40, 400] mg/dL and splits each series at gaps longer than 60 minutes (**no interpolation**);
* `02_split.py` splits **patients** 70/15/15 into train, validation and test, so the test patients are never seen during training;
* `03_windows.py` cuts 6-hour windows, rescales time to [-1, 1] and standardises glucose with training statistics;
* `04_make_tasks.py` generates, once and for all, the fixed evaluation tasks used by every model.

In [ ]:
GNP = os.path.join(REPO, 'glucose_np')
if not os.path.isdir(os.path.join(GNP, 'results')):
    zipfile.ZipFile(os.path.join(GNP, 'results.zip')).extractall(GNP)   # final results, stored zipped

if not os.path.exists(os.path.join(GNP, 'data', 'tasks', 'e1_test.npz')):
    tmp = os.path.join(tempfile.gettempdir(), 'GlucoBench')
    if not os.path.isdir(tmp):
        run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/IrinaStatsLab/GlucoBench.git', tmp])
    raw = os.path.join(GNP, 'data', 'raw'); os.makedirs(raw, exist_ok=True)
    with zipfile.ZipFile(os.path.join(tmp, 'raw_data.zip')) as z:
        for name in z.namelist():
            if name.endswith('.csv') and not name.startswith('__MACOSX'):
                with z.open(name) as src, open(os.path.join(raw, os.path.basename(name)), 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    for step in ['01_preprocess', '02_split', '03_windows', '04_make_tasks']:
        print('>>', step)
        run([sys.executable, f'{step}.py'], cwd=GNP)

Each **window** is an irregularly sampled function: up to 73 readings, with missing ones where the sensor failed. Let's plot a few training windows in mg/dL.

In [ ]:
import config as C
from config import load_windows

train_windows = load_windows('train')
print(f'{len(train_windows)} training windows')

fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
rng = np.random.default_rng(1)
for ax, i in zip(axes, rng.choice(len(train_windows), 3, replace=False)):
    w = train_windows[i]
    ax.plot(w['t_min'] / 60, w['gl'], '.-', ms=4)
    ax.set_title(f"{w['dataset']} · {len(w['x'])} readings")
    ax.set_xlabel('hours')
axes[0].set_ylabel('glucose (mg/dL)')
plt.tight_layout()

### What is a task?

A neural process sees a few **context** points $(x_c, y_c)$ and predicts a distribution over the **target** points $y_t$ at times $x_t$. The batches use the same format as the original TNP/FlowNP code: `batch.xc, batch.yc, batch.xt, batch.yt`, each of shape `[B, N, 1]`.

The fixed evaluation tasks cover four scenarios: random context (E1), context size (E2), realistic sensor gaps and short-term forecasting (E3). Below, one task of each kind from the test set.

In [ ]:
from sampling import load_tasks, TrainSampler, task_batches

def to_mgdl(y):
    return np.asarray(y) * 78.4 + 156.2   # inverse of the z-score (training mean and s.d.)

fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, name in zip(axes, ['e1_test', 'e3_gap120', 'e3_fc60']):
    tasks, meta = load_tasks(name)
    t = tasks[7]
    ax.plot(t['xc'], to_mgdl(t['yc']), 'ko', ms=4, label='context')
    ax.plot(t['xt'], to_mgdl(t['yt']), 'o', color='tab:orange', ms=3, alpha=.6, label='target')
    ax.set_title(f"{name}: Nc={len(t['xc'])}, Nt={len(t['xt'])}")
    ax.set_xlabel('time (normalised)')
axes[0].set_ylabel('glucose (mg/dL)'); axes[0].legend()
plt.tight_layout()

## 3. The models

**CNP** (Garnelo et al., 2018) averages an embedding of the context points into a single vector and predicts an independent Gaussian $\mathcal{N}(\mu(x_t), \sigma^2(x_t))$ at each target. It is fast, but every target point is predicted separately.

**FlowNP** (Hamad & Rosenbaum, 2025) is a transformer that takes the context and a noisy version $y^{(s)}$ of all targets at flow time $s \in [0, 1]$, and predicts a velocity $v_\theta(y^{(s)}, s \mid \text{context})$. It is trained with the flow matching loss: for noise $y^{(0)} \sim \mathcal{N}(0, I)$, the true targets $y^{(1)}$ and a random flow time $s \sim \mathcal{U}[0,1]$ (drawn per target point during training),

$$\mathcal{L} = \mathbb{E}\,\big\| v_\theta\big((1-s)\,y^{(0)} + s\,y^{(1)}, s\big) - (y^{(1)} - y^{(0)}) \big\|^2 .$$

To sample, we integrate $dy/ds = v_\theta(y, s)$ from noise ($s=0$) to data ($s=1$). All target points are generated **jointly and in parallel**, so the trajectories are coherent and need not be Gaussian.

Both models are taken from the official FlowNP repository with its configuration for 1D regression.

In [ ]:
os.chdir(os.path.join(REPO, 'flowNP'))
import yaml
from utils.misc import load_module

def build(model_file, cls, config):
    cfg = yaml.safe_load(open(config)) or {}
    return getattr(load_module(model_file), cls)(**cfg).to(device)

flownp = build('models/fnp.py', 'FNP', 'configs/glucose/fnp.yaml')
cnp = build('models/cnp.py', 'CNP', 'configs/glucose/cnp.yaml')
for name, m in [('FlowNP', flownp), ('CNP', cnp)]:
    print(f'{name}: {sum(p.numel() for p in m.parameters()):,} parameters')

## 4. Training

`TrainSampler` draws random batches from the training windows: for each batch it picks a number of context points $N_c \sim \mathcal{U}[5, 50]$, uses them as context and the rest of the window as target. Both models return `outs.loss` from their forward pass, so the training loop is the same.

The full experiments use 100,000 steps (≈ 25–35 minutes per model on a T4). Here we train for **3,000 steps** to keep the tutorial short, so the models are far from converged.

In [ ]:
def train(model, steps=3000, lr=5e-4, seed=0, log_every=500):
    torch.manual_seed(seed)
    sampler = TrainSampler(train_windows, seed=seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    model.train(); t0 = time.time()
    for step in range(1, steps + 1):
        batch = sampler.sample(batch_size=16, device=device)
        loss = model(batch).loss
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if step % log_every == 0:
            print(f'step {step:5d} | loss {loss.item():8.4f} | {time.time() - t0:5.0f}s')
    model.eval()
    return model

print('FlowNP'); train(flownp)
print('CNP');    train(cnp)

## 5. Sampling trajectories

`glucose.py` contains the sampler we use for FlowNP: a fixed-step ODE solver (`ode_sample`) that turns Gaussian noise into 100 joint samples of the targets. For CNP we sample from its Gaussian predictive.

We take a test window with a **2-hour sensor gap** (`e3_gap120`) and compare the 90% intervals and a few sample trajectories.

In [ ]:
from glucose import ode_sample, ode_loglik

batch = next(iter(task_batches('e3_gap120', batch_size=16, device=device)))
with torch.no_grad():
    fn_samples = ode_sample(flownp, batch, num_samples=100, steps=20, method='midpoint')   # [S, B, Nt, 1]
    py = cnp.predict(batch.xc, batch.yc, batch.xt)
    cnp_samples = py.sample([100])                                                       # [S, B, Nt, 1]

def show(ax, b, samples, title, k=0):
    xt = b.xt[k, :, 0].cpu(); s = to_mgdl(samples[:, k, :, 0].cpu())
    lo, hi = np.percentile(s, [5, 95], axis=0)
    ax.fill_between(xt, lo, hi, alpha=.25, label='90% interval')
    ax.plot(xt, s[:5].T, lw=.8, alpha=.8)
    ax.plot(b.xc[k, :, 0].cpu(), to_mgdl(b.yc[k, :, 0].cpu()), 'ko', ms=3, label='context')
    ax.plot(xt, to_mgdl(b.yt[k, :, 0].cpu()), 'k--', lw=1, label='truth')
    ax.set_title(title); ax.set_xlabel('time (normalised)')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
show(axes[0], batch, fn_samples, 'FlowNP: joint samples')
show(axes[1], batch, cnp_samples, 'CNP: independent Gaussians')
axes[0].set_ylabel('glucose (mg/dL)'); axes[0].legend(loc='upper left')
plt.tight_layout()

FlowNP's samples are smooth curves, because all targets are generated jointly. CNP's samples are independent at each time step, so they look jagged: its marginal intervals can be reasonable, but each individual sample is not a plausible glucose trajectory.

## 6. Scoring with the shared metrics

Every model in the project is scored by the same `Evaluator` (`glucose_np/metrics.py`). It receives predictions in normalised space and reports, in mg/dL:

* **LL**: log-likelihood per target point (higher is better). For FlowNP we compute the joint likelihood with the instantaneous change of variables, integrating the ODE backwards (`ode_loglik`).
* **RMSE / MAE**: error of the point prediction (fidelity).
* **CRPS**: a proper score of the whole predictive distribution, which rewards both accuracy and an appropriate spread.
* **cov90**: fraction of target points inside the 90% interval (0.90 is ideal).

To keep it quick we score only the first 20 batches of `e1_test` (about 150 of its 1,922 tasks), and use `summary()` instead of `save()` so the official results are not overwritten.

In [ ]:
from metrics import Evaluator

def score(name, add, max_batches=20):
    ev = Evaluator(task='e1_test', model='_tutorial', seed=0)
    for i, b in enumerate(task_batches('e1_test', batch_size=32, device=device)):
        if i >= max_batches: break
        with torch.no_grad():
            add(ev, b)
    o = ev.summary()['overall']
    print(f"{name:7s} LL {o['loglik_mgdl']:7.3f} | RMSE {o['rmse_mgdl']:6.2f} | CRPS {o['crps_mgdl']:5.2f} | cov90 {o['cov90']:.3f}")

score('FlowNP', lambda ev, b: ev.add_samples(b, ode_sample(flownp, b, 100, 20, 'midpoint'),
                                            ode_loglik(flownp, b, steps=50)))
score('CNP', lambda ev, b: ev.add_gaussian(b, cnp.predict(b.xc, b.yc, b.xt).loc,
                                          cnp.predict(b.xc, b.yc, b.xt).scale))

## 7. Results of the full experiments

The numbers above come from 3,000 training steps on a subset of tasks. The repository contains the results of the full experiments: 100,000 training steps, 3 seeds, all 11 test scenarios and 5 models (FlowNP, TNP, ANP, CNP and a Matérn Gaussian process), plus two reference baselines. We read them from `glucose_np/results/`.

In [ ]:
RES = os.path.join(REPO, 'glucose_np', 'results')
MODELS = ['FlowNP', 'TNP', 'ANP', 'GP', 'CNP', 'Interp. lineal']

def result(model, task, key):
    vals = [json.load(open(f))['overall'][key] for f in glob.glob(f'{RES}/{model}/seed*/{task}.json')]
    return np.mean(vals), np.std(vals)

print(f"{'e1_test':15s} {'LL':>15s} {'RMSE':>15s} {'CRPS':>15s} {'cov90':>8s}")
for m in MODELS:
    (ll, s1), (rmse, s2), (crps, s3), (cov, _) = [result(m, 'e1_test', k) for k in
                                                  ['loglik_mgdl', 'rmse_mgdl', 'crps_mgdl', 'cov90']]
    print(f'{m:15s} {ll:8.3f} ± {s1:.3f} {rmse:8.2f} ± {s2:.2f} {crps:8.2f} ± {s3:.2f} {cov:8.3f}')

### E2: does FlowNP's advantage grow when context is scarce?

All four E2 tasks share the same windows and targets; only the number of context points changes.

In [ ]:
NCS = [5, 10, 20, 50]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for m in MODELS:
    ll = [result(m, f'e2_nc{n}', 'loglik_mgdl')[0] for n in NCS]
    crps = [result(m, f'e2_nc{n}', 'crps_mgdl')[0] for n in NCS]
    style = dict(lw=3) if m == 'FlowNP' else dict(lw=1.2, alpha=.8)
    axes[0].plot(NCS, ll, 'o-', label=m, **style); axes[1].plot(NCS, crps, 'o-', label=m, **style)
for ax, t in zip(axes, ['LL per point (↑)', 'CRPS, mg/dL (↓)']):
    ax.set_xscale('log'); ax.set_xticks(NCS, NCS); ax.set_xlabel('context points $N_c$'); ax.set_title(t)
axes[1].set_yscale('log'); axes[0].legend(fontsize=8)
plt.tight_layout()

FlowNP has the best log-likelihood at every context size, and its lead over TNP grows from 0.09 nats at $N_c=50$ to 0.27 at $N_c=5$. In CRPS, however, all neural processes converge with 5 context points: with so little information the extra advantage is in the calibration of the distribution, not in the point prediction.

### E4: sampling time

TNP generates the targets one at a time (one network pass per target point), whereas FlowNP integrates an ODE whose cost depends on the number of solver steps but not on the number of targets. Both were timed under identical conditions: T4 GPU, `e1_test`, 100 samples per task, batch 16.

In [ ]:
fl = json.load(open(f'{RES}/FlowNP/e4_timing.json'))['rows']
tn = json.load(open(f'{RES}/TNP/e4_timing.json'))['rows'][0]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot([r['nfe'] for r in fl], [1000 * r['time_per_task_s'] for r in fl], 'o-', label='FlowNP (Euler)')
axes[0].plot(tn['nfe'], 1000 * tn['time_per_task_s'], 's', ms=10, color='tab:red', label='TNP (autoregressive)')
axes[1].plot([1000 * r['time_per_task_s'] for r in fl], [r['crps_mgdl'] for r in fl], 'o-', label='FlowNP (Euler)')
axes[1].plot(1000 * tn['time_per_task_s'], tn['crps_mgdl'], 's', ms=10, color='tab:red', label='TNP')
axes[0].set(xscale='log', yscale='log', xlabel='network evaluations', ylabel='ms per task', title='Cost')
axes[1].set(xscale='log', xlabel='ms per task', ylabel='CRPS (mg/dL)', title='Quality vs time')
axes[0].legend(); plt.tight_layout()

FlowNP reaches TNP's CRPS with very few solver steps and, with the configuration we use (midpoint, 20 steps), samples about **twice as fast** as TNP with a better CRPS.

## Summary

* Neural processes turn imputation and forecasting of CGM series into conditional generation of functions.
* FlowNP generates whole trajectories jointly with flow matching; on real CGM data it gives the best log-likelihood and CRPS among the models we compared, especially on long gaps and 30–60 minute forecasts.
* Linear interpolation is hard to beat in point error on such a smooth signal: the value of the generative models lies in calibrated distributions over plausible trajectories.

All the code that produced the full results is in `glucose_np/` (data, tasks, metrics, figures), `flowNP/glucose.py` (FlowNP), `flowNP/baselines.py` (CNP, ANP, TNP) and `flowNP/gp_matern.py` (Gaussian process).